# Análisis de inasistencias

Inasistencias por programa, sede, semestre, módulo, grupo y seguimiento.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/inasistencias_enriquecido.csv")

print(f"Total registros: {len(df):,}")
print(f"Estudiantes únicos: {df['Numero_identificacion_estudiante'].nunique():,}")
print(f"Módulos: {df['Nombre_modulo'].nunique()}")
print(f"Sedes: {df['Sede'].nunique()}")
print(f"Programas: {df['Nombre_programa_limpio'].nunique()}")

In [ ]:
tabla_programa = df.groupby(["Nombre_programa_limpio", "Sede", "Nombre_nivel"]).agg(
    Total_estudiantes=("Numero_identificacion_estudiante", "nunique"),
    Total_inasistencias=("Cantidad_inasistencia", "sum"),
    Promedio_inasistencias=("Cantidad_inasistencia", "mean")
).reset_index().round(1).sort_values("Promedio_inasistencias", ascending=False).reset_index(drop=True)

display(tabla_programa)

In [ ]:
tabla_modulo = df.groupby(["Nombre_modulo", "Nombre_programa_limpio"]).agg(
    Total_estudiantes=("Numero_identificacion_estudiante", "nunique"),
    Total_inasistencias=("Cantidad_inasistencia", "sum"),
    Promedio_inasistencias=("Cantidad_inasistencia", "mean")
).reset_index().round(1).sort_values("Promedio_inasistencias", ascending=False).reset_index(drop=True)

display(tabla_modulo)

In [ ]:
tabla_seguimiento = df.groupby(["Seguimiento", "Grupo", "Nombre_programa_limpio"]).agg(
    Total_estudiantes=("Numero_identificacion_estudiante", "nunique"),
    Total_inasistencias=("Cantidad_inasistencia", "sum"),
    Promedio_inasistencias=("Cantidad_inasistencia", "mean")
).reset_index().round(1).sort_values("Promedio_inasistencias", ascending=False).reset_index(drop=True)

display(tabla_seguimiento)

In [ ]:
print("=== TOP 10 MÓDULOS CON MAYOR PROMEDIO DE INASISTENCIAS ===")
display(tabla_modulo.head(10))

In [ ]:
print("=== INASISTENCIAS POR SEDE ===")
tabla_sede = df.groupby("Sede").agg(
    Total_estudiantes=("Numero_identificacion_estudiante", "nunique"),
    Total_inasistencias=("Cantidad_inasistencia", "sum"),
    Promedio_inasistencias=("Cantidad_inasistencia", "mean")
).reset_index().round(1).sort_values("Promedio_inasistencias", ascending=False).reset_index(drop=True)

display(tabla_sede)

In [ ]:
print("=== PROMEDIO DE INASISTENCIAS POR ESTUDIANTE, PROGRAMA, SEMESTRE Y SEDE ===\n")

# Por estudiante + seguimiento: contar inasistencias (cada fila = una falta)
est_seg = df.groupby([
    "Numero_identificacion_estudiante", "Nombre_programa_limpio",
    "Nombre_nivel", "Sede", "Seguimiento"
], as_index=False).size().rename(columns={"size": "Inasistencias"})

# Pivot: seguimientos como columnas
pivot = est_seg.pivot_table(
    index=["Nombre_programa_limpio", "Nombre_nivel", "Sede", "Numero_identificacion_estudiante"],
    columns="Seguimiento",
    values="Inasistencias",
    aggfunc="sum",
    fill_value=0,
)
pivot["Total_inasistencias"] = pivot.sum(axis=1)

# Promediar por programa + semestre + sede
columnas_seg = [c for c in pivot.columns if c != "Total_inasistencias"]
promedios = pivot.groupby(["Nombre_programa_limpio", "Nombre_nivel", "Sede"])[columnas_seg + ["Total_inasistencias"]].mean().round(1).reset_index()

promedios = promedios.sort_values("Total_inasistencias", ascending=False).reset_index(drop=True)

display(promedios)